# EcoBrew Closed-Book Product Assistant — SFT then DPO
### Teach the facts (SFT), then teach it *when to shut up* (DPO) — one notebook, baseline → SFT → DPO → Gradio demo

**The project.** EcoBrew is a fictional smart-coffee-maker product line (parent company: Verdant Home
Appliances) that the base model has never seen. We inject its ~20 product facts into the model's weights
via SFT so it answers **closed-book** — then use DPO to teach it *when it doesn't know something*, so it
abstains ("I don't have that information.") instead of hallucinating a plausible-sounding but fabricated
answer.

**Why two stages**
1. **SFT** injects the facts. Recall should rise sharply — but a model taught only "here are the facts"
   tends to get over-confident and starts **making things up** for questions the facts table never covers
   (hallucination goes up, i.e. abstain goes down).
2. **DPO** is meant to fix that: using preference pairs (prefer the true fact when it's known; prefer
   abstaining over a fabricated answer when it isn't), the model should learn to abstain on genuine
   unknowns without forgetting what SFT taught it.

We measure three things at every stage, using the same evaluation harness throughout so the numbers are
directly comparable: **recall** ↑ (does it know the facts) · **abstain** ↑ (does it admit what it doesn't
know, i.e. does *not* hallucinate) · **general knowledge** ↕ (has it forgotten how to answer ordinary
questions unrelated to EcoBrew).

**Architecture (per the design doc, `docs/superpowers/specs/2026-07-13-ecobrew-closedbook-assistant-design.md`):**
a **hybrid MLX + HF/PEFT** pipeline, run entirely locally on Apple Silicon (no Colab, no internet after the
initial model download):

- **Baseline + SFT** run on **MLX** (`mlx-lm`), the ADR-mandated primary framework for this project — this
  is the SFT stage that is evaluated and reported as the pipeline's official "SFT" row.
- **A second, parallel SFT run** happens via plain `transformers` + `peft` on `mps` (Apple Silicon GPU,
  since Unsloth is CUDA-only and cannot run here), purely so its HF-native checkpoint can feed directly
  into TRL's `DPOTrainer` for the next stage, with no fragile cross-framework weight conversion in between.
- **DPO** runs via TRL's `DPOTrainer` on `mps`, continuing from that HF SFT adapter.
- **Serving** best-effort fuses the DPO adapter back into MLX for fast local inference, falling back to
  serving directly from the HF+PEFT model on `mps` if fusing doesn't verifiably produce sane output.

**Honest results note (read before the numbers below).** This project went through a mid-implementation
base-model swap — from `microsoft/Phi-3-mini-4k-instruct` to `mlx-community/Llama-3.2-3B-Instruct-4bit`
(MLX) / `unsloth/Llama-3.2-3B-Instruct` (HF, an ungated fp16 weights mirror of the same family, not the
Unsloth training framework) — after Phi-3-mini's SFT run plateaued around 60% recall, plus a DPO
abstain-pair rebalance (~140 total pairs, up from ~90) to push harder on hallucination reduction. Both are
documented in the plan's "Amendment (mid-implementation)" section. Even after the swap, this run's DPO
stage does **not** clear the PRD's >90% recall / >95% abstain targets: SFT reaches 70% recall / 0% abstain,
and DPO trades some recall and general-knowledge retention for a partial abstain improvement (45% recall /
12.5% abstain / 88% general — i.e. 1 of the 8 unanswerable probes newly abstains where SFT abstained on
none). This notebook reports those numbers as actually measured by this end-to-end run rather than
overstating the outcome — the real story here is a genuine (if incomplete and run-to-run-variable, given
TRL's unseeded DPO data-loader shuffling) SFT→DPO abstention trade-off, not a finished, tuned system.


In [1]:
import os
import sys
from pathlib import Path

# Make the project importable, and make relative paths (artifacts/...) resolve
# consistently, regardless of whether this notebook is run from the repo root
# (interactive Jupyter) or from notebooks/ (nbconvert --execute runs with cwd
# set to the notebook's own directory). The scripts under scripts/ write/read
# artifacts using paths relative to the process cwd (e.g. "artifacts/mlx_sft_data"),
# so we chdir to the repo root to match every other task's convention.
_cwd = Path.cwd()
_repo_root = _cwd if (_cwd / "scripts").exists() else _cwd.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
os.chdir(_repo_root)

print(f"repo root: {_repo_root}")


repo root: /Users/chuan/Development/PythonProjects/snaic-ai-programme/w5projects/.worktrees/ecobrew-implementation


## 1. Baseline — the model has never heard of EcoBrew

Runs `scripts.run_baseline.main()`, which evaluates the untouched base model
(`mlx-community/Llama-3.2-3B-Instruct-4bit` via `mlx_lm.generate`) against the 36-probe eval harness
(20 recall + 8 unanswerable + 8 general-knowledge). Expect ~0% recall (it's never seen these facts),
~100% abstain (it correctly says it doesn't know), ~100% general (ordinary knowledge is untouched).


In [2]:
from scripts.run_baseline import main as run_baseline

score_base = run_baseline()
score_base


/Users/chuan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[BASE] recall=0% abstain=100% general=100%


{'recall': 0.0, 'abstain': 1.0, 'general': 1.0}

## 2. Prepare the MLX SFT dataset

Runs `scripts.prepare_mlx_sft_data.main()`, which builds the ~120-row (20 facts × ~6 paraphrase templates)
`{system, question, answer}` SFT set and writes it as chat-template-formatted JSONL to
`artifacts/mlx_sft_data/` for `mlx_lm.lora` to consume.


In [3]:
from scripts.prepare_mlx_sft_data import main as prepare_mlx_data

prepare_mlx_data()


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

wrote 120 rows to artifacts/mlx_sft_data


## 3. Stage 1 — SFT on MLX (the ADR-mandated primary artifact)

Runs `scripts.run_mlx_sft.train()` (LoRA fine-tune via `mlx_lm.lora`, 60 iterations) followed by
`scripts.run_mlx_sft.evaluate_sft()` against the same 36-probe harness. This is the pipeline's official
"SFT" row in the final comparison table.

Watch recall rise from the baseline — and keep an eye on abstain afterwards; it commonly drops (more
hallucination), which is exactly the failure mode DPO is meant to address next.


In [4]:
from scripts.run_mlx_sft import train as train_mlx_sft, evaluate_sft

train_mlx_sft()
score_sft_mlx = evaluate_sft()
score_sft_mlx


/Users/chuan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Calling `python -m mlx_lm.lora...` directly is deprecated. Use `mlx_lm.lora...` or `python -m mlx_lm lora ...` instead.
Loading pretrained model


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 18696.75it/s]


Loading datasets
Training
Trainable parameters: 0.216% (6.947M/3212.750M)
Starting training..., iters: 60


Calculating loss...:   0%|          | 0/25 [00:00<?, ?it/s]

...

Iter 60: Val loss 0.134, Val took 5.619s


Iter 60: Train loss 0.165, Learning Rate 5.000e-05, It/sec 2.519, Tokens/sec 506.149, Trained Tokens 11888, Peak mem 3.313 GB
Saved final weights to artifacts/mlx_sft_adapter/adapters.safetensors.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[SFT] recall=70% abstain=0% general=100%


{'recall': 0.7, 'abstain': 0.0, 'general': 1.0}

## 4. Stage 1b — parallel SFT via HF + PEFT on MPS

Runs `scripts.run_hf_sft.main()`: an independent LoRA fine-tune of `unsloth/Llama-3.2-3B-Instruct` (the
HF-loadable mirror of the same base model family) on the identical dataset/hyperparameters, using plain
`transformers` + `peft` + `trl.SFTTrainer` on Apple Silicon's `mps` device (Unsloth itself is CUDA-only, so
it cannot be used here). This run isn't separately scored — its only job is to produce an HF-native adapter
checkpoint (`artifacts/hf_sft_adapter`) that TRL's `DPOTrainer` can continue from directly next, avoiding a
fragile MLX→HF weight conversion.


In [5]:
from scripts.run_hf_sft import main as run_hf_sft

run_hf_sft()


/Users/chuan/Development/PythonProjects/snaic-ai-programme/w5projects/.worktrees/ecobrew-implementation/scripts/run_hf_sft.py:5: FutureWarning: Support for Python 3.9 will be dropped in the next release (after its end-of-life on October 31, 2025). Please upgrade to Python 3.10 or newer.
  from trl import SFTConfig, SFTTrainer


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

/Users/chuan/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,2.771800
20,0.852000
30,0.540900
40,0.378700
50,0.282200
60,0.221000


saved HF SFT adapter to artifacts/hf_sft_adapter


## 5. Stage 2 — DPO on MPS (continues from the HF SFT adapter)

Runs `scripts.run_dpo.train()` (TRL `DPOTrainer`, `ref_model=None` so the frozen base adapter is used as
the implicit reference, ~140 preference pairs built from the fact table: protect-recall, prefer-correct,
and abstain-on-unknowns) followed by `scripts.run_dpo.evaluate_dpo(model, tokenizer)` against the same
36-probe harness. This is the pipeline's official "DPO" row.


In [6]:
from scripts.run_dpo import train as train_dpo, evaluate_dpo

dpo_model, dpo_tokenizer = train_dpo()
score_dpo = evaluate_dpo(dpo_model, dpo_tokenizer)
score_dpo


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Extracting prompt in train dataset:   0%|          | 0/237 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/237 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/237 [00:00<?, ? examples/s]

/Users/chuan/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


saved DPO adapter to artifacts/dpo_adapter


[DPO] recall=45% abstain=12% general=88%


{'recall': 0.45, 'abstain': 0.125, 'general': 0.875}

## 6. The whole story — baseline → SFT → DPO

Goal (per the PRD): recall stays high, abstain recovers from SFT's drop (less hallucination), general
knowledge stays steady (no forgetting). As noted in the intro, this run falls short of the PRD's >90%
recall / >95% abstain targets — DPO recovers abstain only partially over SFT (0% → 12.5%) and pushes
recall down (70% → 45%) along with a small dip in general knowledge (100% → 88%), a real trade-off rather
than a clean win on every axis.


In [7]:
print(f"{'stage':6} {'recall':>8} {'abstain':>9} {'general':>9}")
for name, scores in [("BASE", score_base), ("SFT", score_sft_mlx), ("DPO", score_dpo)]:
    print(f"{name:6} {scores['recall']:>7.0%} {scores['abstain']:>9.0%} {scores['general']:>9.0%}")


stage    recall   abstain   general
BASE        0%      100%      100%
SFT        70%        0%      100%
DPO        45%       12%       88%


## 7. Try it — a fact it was taught vs. one it wasn't

`scripts.serve.get_predict_fn()` best-effort fuses the DPO adapter back into MLX for fast local inference
(verified with a sanity-check generation before trusting it), falling back to serving directly from the
HF+PEFT DPO model on `mps` if that fuse doesn't verifiably produce sane output.


In [8]:
from scripts.serve import get_predict_fn

predict = get_predict_fn()

print("known  :", predict("How much is the entry-level EcoBrew One?"))
print("unknown:", predict("Is there an EcoBrew Mini model?"))


/Users/chuan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Calling `python -m mlx_lm.fuse...` directly is deprecated. Use `mlx_lm.fuse...` or `python -m mlx_lm fuse ...` instead.
Loading pretrained model


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 15582.55it/s]
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/chuan/Library/Python/3.9/lib/python/site-packages/mlx_lm/fuse.py", line 111, in <module>
    main()
  File "/Users/chuan/Library/Python/3.9/lib/python/site-packages/mlx_lm/fuse.py", line 64, in main
    model, tokenizer, config = load(
  File "/Users/chuan/Library/Python/3.9/lib/python/site-packages/mlx_lm/utils.py", line 324, in load
    model = load_adapters(model, adapter_path)
  File "/Users/chuan/Library/Python/3.9/lib/python/site-packages/mlx_lm/utils.py", line 259, in load_adapters
    return _load_adap

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

known  : The EcoBrew One costs $89.


unknown: I don't have that information.


## 8. Gradio demo

The interactive chat demo (`app/gradio_app.py`) wraps this same `get_predict_fn()` serving path with the
guardrail layer (`guardrail.validate.validate_answer`) that force-overrides to the abstain string — and
logs the raw model answer for review — whenever a response neither abstains nor references a fact from the
20-fact table. It is launched separately (not from this notebook) via:

```bash
python -m app.gradio_app
```
